<a href="https://colab.research.google.com/github/martirossi/AppliedML2026_mr/blob/main/clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

# ============================
# Load dataset
# ============================

data_path = "/content/drive/MyDrive/Colab Notebooks/SDSS-Gaia_5950stars.csv"
df = pd.read_csv(data_path)

# ============================
# Two feature sets (max 6 variables)
# ============================

set1 = ["C_FE", "N_FE", "O_FE", "MG_FE", "FE_H", "NI_FE"]
set2 = ["J", "K", "SI_FE", "CA_FE", "Energy", "Lz"]

# Keep only available columns
set1 = [f for f in set1 if f in df.columns]
set2 = [f for f in set2 if f in df.columns]

# ============================
# Helper functions
# ============================

def find_best_k(X, k_min=4, k_max=40, patience=5):
    best_k = None
    best_score = -1
    no_improve = 0

    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(X)
        score = silhouette_score(X, labels)

        if score > best_score + 1e-4:
            best_score = score
            best_k = k
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    return best_k, best_score


def evaluate_feature_set(df, features):
    X = df[features].values
    X = StandardScaler().fit_transform(X)

    best_k, best_sil = find_best_k(X)

    # Cross-validation silhouette
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in kf.split(X):
        km = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
        km.fit(X[train_idx])
        labels = km.predict(X[val_idx])
        cv_scores.append(silhouette_score(X[val_idx], labels))

    return {
        "features": features,
        "best_k": best_k,
        "cv_mean": np.mean(cv_scores),
        "X_scaled": X
    }


def fit_final_model(X, k):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X)
    return labels

# ============================
# Evaluate both feature sets
# ============================

res1 = evaluate_feature_set(df, set1)
res2 = evaluate_feature_set(df, set2)

print("Set 1 CV silhouette:", res1["cv_mean"])
print("Set 2 CV silhouette:", res2["cv_mean"])

# ============================
# Choose the best set
# ============================

best = res1 if res1["cv_mean"] > res2["cv_mean"] else res2

print("\nBest feature set:", best["features"])
print("Best k:", best["best_k"])

# ============================
# Fit final model and save files
# ============================

final_labels = fit_final_model(best["X_scaled"], best["best_k"])

# Convert labels to 1..k instead of 0..k-1
final_labels = final_labels + 1

# Save variable list (one feature per line, no header)
varlist_path = "/content/drive/MyDrive/Colab Notebooks/Clustering_MartinaRossi_KMeans_VariableList.csv"
with open(varlist_path, "w") as f:
    for feat in best["features"]:
        f.write(f"{feat}\n")

# Save clustering labels (one label per line, no header)
labels_path = "/content/drive/MyDrive/Colab Notebooks/Clustering_MartinaRossi_KMeans.csv"
with open(labels_path, "w") as f:
    for lab in final_labels:
        f.write(f"{lab}\n")

print("\nSaved:")
print(" -", varlist_path)
print(" -", labels_path)


Set 1 CV silhouette: 0.23796832910885596
Set 2 CV silhouette: 0.24827519662495456

Best feature set: ['J', 'K', 'SI_FE', 'CA_FE', 'Energy', 'Lz']
Best k: 5

Saved:
 - /content/drive/MyDrive/Colab Notebooks/Clustering_MartinaRossi_KMeans_VariableList.csv
 - /content/drive/MyDrive/Colab Notebooks/Clustering_MartinaRossi_KMeans.csv
